# MVP Pipeline: Adaptive Augmentations for Small Objects

This notebook is the main entrypoint for testing MVP in Google Colab.

Pipeline steps:
1. Install dependencies.
2. Validate dataset structure.
3. Analyze dataset and save stats.
4. Generate adaptive augmentation policy.
5. (Optional) Run training modes.
6. (Optional) Convert to COCO and run COCOeval.

In [ ]:
# Install dependencies in Colab runtime.
%pip install -q ultralytics albumentations opencv-python pycocotools pyyaml numpy pandas matplotlib tqdm gdown


In [ ]:
from pathlib import Path
import sys

# 1) Попробуем найти проект в типовых местах
candidates = [
    Path("/content/small_objects_auto_aug"),
    Path("/content/drive/MyDrive/small_objects_auto_aug"),
    Path.cwd(),
    Path.cwd().parent,
]

PROJECT_ROOT = None
for p in candidates:
    if (p / "src").exists() and (p / "configs").exists():
        PROJECT_ROOT = p
        break

# 2) Если не нашли — клонируем в /content (подставь свой URL)
if PROJECT_ROOT is None:
    import subprocess
    repo_url = "https://github.com/s44w/small_objects_auto_aug.git"  # <-- замени
    subprocess.run(["git", "clone", repo_url, "/content/small_objects_auto_aug"], check=True)
    PROJECT_ROOT = Path("/content/small_objects_auto_aug")

sys.path.insert(0, str(PROJECT_ROOT))
print("Using project root:", PROJECT_ROOT)


In [ ]:
# Helpers for robust dataset path checks.
from pathlib import Path

def has_yolo_structure(root: Path) -> bool:
    req = [
        root / 'images' / 'train',
        root / 'images' / 'val',
        root / 'labels' / 'train',
        root / 'labels' / 'val',
    ]
    return all(p.exists() for p in req)


In [ ]:
# Imports and configuration.
from src.utils.io import load_yaml
from src.data.visdrone_manager import validate_visdrone_yolo_structure, save_validation_report
from src.analysis.dataset_analyzer import DatasetAnalyzerConfig, analyze_dataset
from src.policy.rule_engine import RuleEngineConfig, generate_policy_from_stats, save_policy_artifacts

project_cfg = load_yaml(PROJECT_ROOT / 'configs' / 'project_config.yaml')

# Keep config root as fallback only; actual dataset_root is resolved in bootstrap cell.
dataset_root_cfg = Path(project_cfg['dataset']['root'])
if not dataset_root_cfg.is_absolute():
    dataset_root_cfg = PROJECT_ROOT / dataset_root_cfg

splits = tuple(project_cfg['dataset'].get('splits', ['train', 'val']))
image_extensions = tuple(project_cfg['dataset'].get('image_extensions', ['.jpg', '.jpeg', '.png', '.bmp']))

stats_dir = PROJECT_ROOT / 'artifacts' / 'stats'
policy_dir = PROJECT_ROOT / 'artifacts' / 'policy'
stats_dir.mkdir(parents=True, exist_ok=True)
policy_dir.mkdir(parents=True, exist_ok=True)

print('Config dataset root (fallback):', dataset_root_cfg)


In [ ]:
# Optional legacy hotfix cell.
# By default this is disabled because src.data.visdrone_manager already has robust fallback conversion.
APPLY_LEGACY_HOTFIX = False

if APPLY_LEGACY_HOTFIX:
    from pathlib import Path
    import shutil
    import src.data.visdrone_manager as vm

    def _resolve_split(root: Path, split: str):
        names = {
            'train': ['VisDrone2019-DET-train', 'train'],
            'val': ['VisDrone2019-DET-val', 'val'],
            'test': ['VisDrone2019-DET-test-dev', 'VisDrone2019-DET-test-challenge', 'test'],
        }[split]
        for n in names:
            p = root / n
            if p.exists() and p.is_dir():
                return p
        for n in names:
            for p in root.rglob(n):
                if p.is_dir():
                    return p
        # Fallback: allow flat root/images + root/annotations as train source.
        if split == 'train' and (root / 'images').exists() and (root / 'annotations').exists():
            return root
        return None

    def _fallback_prepare_visdrone_auto(raw_visdrone_root, output_root):
        from PIL import Image
        raw_root = Path(raw_visdrone_root)
        out = Path(output_root)
        out.mkdir(parents=True, exist_ok=True)

        for split in ('train', 'val', 'test'):
            src = _resolve_split(raw_root, split)
            if src is None:
                continue

            src_img = src / 'images'
            src_ann = src / 'annotations'
            dst_img = out / 'images' / split
            dst_lbl = out / 'labels' / split
            dst_img.mkdir(parents=True, exist_ok=True)
            dst_lbl.mkdir(parents=True, exist_ok=True)

            for ip in src_img.glob('*.jpg'):
                shutil.copy2(ip, dst_img / ip.name)
            for ip in src_img.glob('*.png'):
                shutil.copy2(ip, dst_img / ip.name)

            if src_ann.exists():
                for ap in src_ann.glob('*.txt'):
                    img = dst_img / f"{ap.stem}.jpg"
                    if not img.exists():
                        img = dst_img / f"{ap.stem}.png"
                    if not img.exists():
                        continue

                    with Image.open(img) as im:
                        w, h = im.width, im.height

                    lines = []
                    for raw in ap.read_text(encoding='utf-8').splitlines():
                        if not raw.strip():
                            continue
                        row = [x.strip() for x in raw.split(',')]
                        if len(row) < 6 or row[4] == '0':
                            continue

                        x, y, bw, bh = map(float, row[:4])
                        cls = int(float(row[5])) - 1
                        if cls < 0:
                            continue

                        xc = (x + bw / 2) / max(1, w)
                        yc = (y + bh / 2) / max(1, h)
                        wn = bw / max(1, w)
                        hn = bh / max(1, h)
                        lines.append(f"{cls} {xc:.6f} {yc:.6f} {wn:.6f} {hn:.6f}\n")

                    (dst_lbl / ap.name).write_text(''.join(lines), encoding='utf-8')

        print(f'[HOTFIX] Fallback conversion complete: {out}')

    vm.prepare_visdrone_auto = _fallback_prepare_visdrone_auto
    prepare_visdrone_auto = vm.prepare_visdrone_auto
    print('Legacy HOTFIX applied: prepare_visdrone_auto patched')
else:
    print('Legacy HOTFIX is disabled (recommended).')



In [ ]:
from pathlib import Path
import zipfile
import shutil
import tempfile
import subprocess
import sys
from google.colab import drive

# Step 0.5 (optional): download official VisDrone train/val ZIPs to Drive and unpack raw layout.
# Disabled by default: if dataset is already prepared in Drive, skip this cell.
RUN_FETCH_AND_EXTRACT = False

drive.mount('/content/drive', force_remount=False)

ZIP_DIR = Path('/content/drive/MyDrive/datasets/visdrone_zips')
RAW_DIR = Path('/content/drive/MyDrive/datasets/visdrone_raw')
ZIP_DIR.mkdir(parents=True, exist_ok=True)
RAW_DIR.mkdir(parents=True, exist_ok=True)

files = {
    'VisDrone2019-DET-train.zip': '1a2oHjcEcwXP8oUF95qiwrqzACb2YlUhn',
    'VisDrone2019-DET-val.zip': '1bxK5zgLn0_L8x276eKkuYA_FzwCIjb59',
}


def _find_split_root(root: Path) -> Path | None:
    if (root / 'images').exists() and (root / 'annotations').exists():
        return root

    for p in root.rglob('*'):
        if p.is_dir() and (p / 'images').exists() and (p / 'annotations').exists():
            return p
    return None


def _split_ready(raw_root: Path, split_stem: str) -> bool:
    split_dir = raw_root / split_stem
    images_dir = split_dir / 'images'
    ann_dir = split_dir / 'annotations'
    if not images_dir.exists() or not ann_dir.exists():
        return False
    image_count = len(list(images_dir.glob('*.jpg'))) + len(list(images_dir.glob('*.png')))
    ann_count = len(list(ann_dir.glob('*.txt')))
    return image_count > 0 and ann_count > 0


def extract_split(zip_path: Path, raw_root: Path):
    target = raw_root / zip_path.stem
    if _split_ready(raw_root, zip_path.stem):
        print(f'[SKIP] ready: {target}')
        return

    target.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory() as td:
        tmp = Path(td)
        with zipfile.ZipFile(zip_path, 'r') as zf:
            zf.extractall(tmp)

        src = _find_split_root(tmp)
        if src is None:
            raise FileNotFoundError(
                f'Could not find images/annotations inside extracted archive: {zip_path}'
            )

        if target.exists():
            shutil.rmtree(target, ignore_errors=True)
        target.mkdir(parents=True, exist_ok=True)

        for item in src.iterdir():
            shutil.move(str(item), str(target / item.name))


if RUN_FETCH_AND_EXTRACT:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'gdown'])
    import gdown

    for name, fid in files.items():
        z = ZIP_DIR / name
        if not z.exists():
            print(f'[DOWNLOAD] {name}')
            gdown.download(id=fid, output=str(z), quiet=False)
        else:
            print(f'[SKIP] zip exists: {z.name}')

    for zname in files.keys():
        extract_split(ZIP_DIR / zname, RAW_DIR)

    train_images = RAW_DIR / 'VisDrone2019-DET-train' / 'images'
    train_ann = RAW_DIR / 'VisDrone2019-DET-train' / 'annotations'
    val_images = RAW_DIR / 'VisDrone2019-DET-val' / 'images'
    val_ann = RAW_DIR / 'VisDrone2019-DET-val' / 'annotations'

    print('train/images exists:', train_images.exists())
    print('train/annotations exists:', train_ann.exists())
    print('val/images exists:', val_images.exists())
    print('val/annotations exists:', val_ann.exists())
else:
    print('Step skipped. Set RUN_FETCH_AND_EXTRACT=True only when raw ZIP bootstrap is needed.')


In [ ]:
# Dataset bootstrap: prefer ready YOLO dataset in Drive, rebuild from raw only on demand.
from pathlib import Path
import shutil
import json
import time
from datetime import datetime, timezone

from google.colab import drive
import importlib
import src.data.visdrone_manager as visdrone_manager_module

importlib.reload(visdrone_manager_module)
prepare_visdrone_auto = visdrone_manager_module.prepare_visdrone_auto
validate_visdrone_yolo_structure = visdrone_manager_module.validate_visdrone_yolo_structure

PROJECT_ROOT = PROJECT_ROOT if 'PROJECT_ROOT' in globals() else Path('/content/small_objects_auto_aug')
DEFAULT_ROOT = PROJECT_ROOT / 'datasets' / 'visdrone_yolo'

drive.mount('/content/drive', force_remount=False)
DRIVE_ROOT = Path('/content/drive/MyDrive')
dataset_root_cfg = globals().get('dataset_root_cfg', None)

# Runtime controls.
FORCE_REBUILD_FROM_RAW = False
AUTO_BUILD_FROM_RAW_IF_YOLO_MISSING = False
USE_LOCAL_RUNTIME_COPY = True
LOCAL_RUNTIME_ROOT = Path('/content/datasets/visdrone_yolo')

# Where to store experiment artifacts in Drive.
DRIVE_ARTIFACTS_ROOT = DRIVE_ROOT / 'experiments' / 'small_objects_auto_aug' / 'visdrone'
SYNC_RESULTS_TO_DRIVE = True

IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp'}
MIN_TRAIN_IMAGES_WARN = 3000
MIN_VAL_IMAGES_WARN = 500


def count_images(images_dir: Path) -> int:
    if not images_dir.exists():
        return 0
    return sum(1 for p in images_dir.iterdir() if p.is_file() and p.suffix.lower() in IMG_EXTS)


def count_labels(labels_dir: Path) -> int:
    if not labels_dir.exists():
        return 0
    return len(list(labels_dir.glob('*.txt')))


def split_counts(root: Path) -> tuple[int, int, int, int]:
    train_img = count_images(root / 'images' / 'train')
    val_img = count_images(root / 'images' / 'val')
    train_lbl = count_labels(root / 'labels' / 'train')
    val_lbl = count_labels(root / 'labels' / 'val')
    return train_img, val_img, train_lbl, val_lbl


def has_yolo_structure(root: Path, require_non_empty: bool = True) -> bool:
    req = [
        root / 'images' / 'train',
        root / 'images' / 'val',
        root / 'labels' / 'train',
        root / 'labels' / 'val',
    ]
    if not all(p.exists() for p in req):
        return False

    if not require_non_empty:
        return True

    train_img, val_img, train_lbl, val_lbl = split_counts(root)
    return train_img > 0 and val_img > 0 and train_lbl > 0 and val_lbl > 0


def find_yolo_root() -> Path | None:
    candidates = [
        DRIVE_ROOT / 'datasets' / 'visdrone_yolo',
        DRIVE_ROOT / 'visdrone_yolo',
        DEFAULT_ROOT,
        Path(dataset_root_cfg) if dataset_root_cfg is not None else None,
    ]
    for c in candidates:
        if c is not None and c.exists() and has_yolo_structure(c, require_non_empty=True):
            return c

    datasets_root = DRIVE_ROOT / 'datasets'
    if datasets_root.exists():
        for cand in datasets_root.iterdir():
            if cand.is_dir() and has_yolo_structure(cand, require_non_empty=True):
                return cand

    return None


def resolve_raw_root() -> Path | None:
    candidates = [
        DRIVE_ROOT / 'datasets' / 'visdrone_raw',
        DRIVE_ROOT / 'visdrone_raw',
    ]
    for c in candidates:
        if not c.exists():
            continue
        train_ok = (c / 'VisDrone2019-DET-train' / 'images').exists() and (c / 'VisDrone2019-DET-train' / 'annotations').exists()
        val_ok = (c / 'VisDrone2019-DET-val' / 'images').exists() and (c / 'VisDrone2019-DET-val' / 'annotations').exists()
        if train_ok and val_ok:
            return c
    return None


def build_yolo_from_raw(raw_root: Path, out_root: Path) -> Path:
    print(f'[INFO] Converting raw VisDrone -> YOLO: {raw_root} -> {out_root}')
    out_root.mkdir(parents=True, exist_ok=True)

    for split in ('train', 'val'):
        shutil.rmtree(out_root / 'images' / split, ignore_errors=True)
        shutil.rmtree(out_root / 'labels' / split, ignore_errors=True)

    prepare_visdrone_auto(raw_visdrone_root=raw_root, output_root=out_root)

    if not has_yolo_structure(out_root, require_non_empty=True):
        train_img, val_img, train_lbl, val_lbl = split_counts(out_root)
        raise RuntimeError(
            'YOLO conversion completed but dataset is incomplete: '
            f'train_img={train_img}, val_img={val_img}, train_lbl={train_lbl}, val_lbl={val_lbl}. '
            'Check raw split layout in Drive.'
        )

    return out_root


def mirror_to_local_runtime(src_root: Path, dst_root: Path) -> Path:
    dst_root.parent.mkdir(parents=True, exist_ok=True)

    src_train_img, src_val_img, src_train_lbl, src_val_lbl = split_counts(src_root)
    src_signature = {
        'src_root': src_root.as_posix(),
        'train_img': src_train_img,
        'val_img': src_val_img,
        'train_lbl': src_train_lbl,
        'val_lbl': src_val_lbl,
    }

    meta_path = dst_root / '_mirror_meta.json'
    if has_yolo_structure(dst_root, require_non_empty=True) and meta_path.exists():
        try:
            old_meta = json.loads(meta_path.read_text(encoding='utf-8'))
        except Exception:
            old_meta = {}
        if all(old_meta.get(k) == v for k, v in src_signature.items()):
            print(f'[OK] Reusing local runtime mirror: {dst_root}')
            return dst_root

    print(f'[INFO] Mirroring dataset to local runtime SSD: {src_root} -> {dst_root}')
    if dst_root.exists() or dst_root.is_symlink():
        if dst_root.is_symlink() or dst_root.is_file():
            dst_root.unlink()
        else:
            shutil.rmtree(dst_root, ignore_errors=True)

    shutil.copytree(src_root, dst_root)
    meta_payload = {
        **src_signature,
        'updated_utc': datetime.now(timezone.utc).isoformat(),
    }
    meta_path.write_text(json.dumps(meta_payload, ensure_ascii=False, indent=2), encoding='utf-8')
    return dst_root


def safe_validate(root: Path, retries: int = 3, delay_sec: float = 2.0):
    last_exc = None
    for attempt in range(1, retries + 1):
        try:
            return validate_visdrone_yolo_structure(root, splits=('train', 'val'))
        except OSError as exc:
            last_exc = exc
            print(f'[WARN] Deep validation failed ({attempt}/{retries}): {exc}')
            if attempt < retries:
                time.sleep(delay_sec * attempt)

    train_img, val_img, train_lbl, val_lbl = split_counts(root)
    return {
        'dataset_root': root.as_posix(),
        'is_valid': (train_img > 0 and val_img > 0 and train_lbl > 0 and val_lbl > 0),
        'warning': f'validate_visdrone_yolo_structure failed: {last_exc}',
        'splits': {
            'train': {'num_images': train_img, 'num_label_files': train_lbl},
            'val': {'num_images': val_img, 'num_label_files': val_lbl},
        },
    }


drive_dataset_root = find_yolo_root()

if drive_dataset_root is None:
    if FORCE_REBUILD_FROM_RAW or AUTO_BUILD_FROM_RAW_IF_YOLO_MISSING:
        raw_root = resolve_raw_root()
        if raw_root is None:
            raise FileNotFoundError(
                'Ready YOLO dataset not found and raw source is incomplete.\n'
                'Expected one of: /content/drive/MyDrive/datasets/visdrone_yolo (ready YOLO)\n'
                'or raw with split dirs: /content/drive/MyDrive/datasets/visdrone_raw/VisDrone2019-DET-{train,val}/{images,annotations}\n'
                'Tip: run the optional ZIP bootstrap cell first.'
            )
        drive_dataset_root = build_yolo_from_raw(raw_root, DRIVE_ROOT / 'datasets' / 'visdrone_yolo')
    else:
        raise FileNotFoundError(
            'Ready YOLO dataset not found in Drive.\n'
            'Place processed data at /content/drive/MyDrive/datasets/visdrone_yolo\n'
            'or set AUTO_BUILD_FROM_RAW_IF_YOLO_MISSING=True to rebuild from raw.'
        )

if USE_LOCAL_RUNTIME_COPY:
    dataset_root = mirror_to_local_runtime(drive_dataset_root, LOCAL_RUNTIME_ROOT)
else:
    dataset_root = drive_dataset_root

DEFAULT_ROOT.parent.mkdir(parents=True, exist_ok=True)
if dataset_root != DEFAULT_ROOT:
    if DEFAULT_ROOT.exists() or DEFAULT_ROOT.is_symlink():
        try:
            if DEFAULT_ROOT.is_symlink() or DEFAULT_ROOT.is_file():
                DEFAULT_ROOT.unlink()
            else:
                shutil.rmtree(DEFAULT_ROOT, ignore_errors=True)
        except Exception as exc:
            print(f'[WARN] Could not clear default root: {exc}')

    try:
        DEFAULT_ROOT.symlink_to(dataset_root, target_is_directory=True)
        print(f'[OK] Linked {DEFAULT_ROOT} -> {dataset_root}')
    except Exception as exc:
        print(f'[WARN] Symlink not created: {exc}')

report = safe_validate(dataset_root)
train_n = report['splits']['train']['num_images']
val_n = report['splits']['val']['num_images']
print('is_valid:', report['is_valid'])
print('train images:', train_n)
print('val images:', val_n)
print('Using dataset_root:', dataset_root)
print('Drive source dataset root:', drive_dataset_root)
print('Drive artifacts root:', DRIVE_ARTIFACTS_ROOT)
if 'warning' in report:
    print('[WARN]', report['warning'])

if train_n < MIN_TRAIN_IMAGES_WARN or val_n < MIN_VAL_IMAGES_WARN:
    print(
        '[WARN] Dataset is smaller than expected full VisDrone split. '
        f'train={train_n}, val={val_n}. This is OK for quick runs, but metrics may be lower.'
    )

assert train_n > 0 and val_n > 0, 'train/val images must be non-empty before training'


In [ ]:
# Quick smoke check (fast): subset -> analyze -> policy -> artifact assertions.
# This is enough to verify project operability without heavy training/eval.
from pathlib import Path
import json

from src.data.subset_builder import build_yolo_subset
from src.analysis.dataset_analyzer import DatasetAnalyzerConfig, analyze_dataset
from src.policy.rule_engine import RuleEngineConfig, generate_policy_from_stats, save_policy_artifacts
from src.data.visdrone_manager import validate_visdrone_yolo_structure

RUN_SMOKE = True

if RUN_SMOKE:
    SMOKE_TRAIN_IMAGES = 60
    SMOKE_VAL_IMAGES = 20
    SMOKE_SPLITS = ('train',)
    SMOKE_SEED = 42

    subset_root = PROJECT_ROOT / 'datasets' / 'visdrone_smoke'
    subset_report = build_yolo_subset(
        dataset_root=dataset_root,
        output_root=subset_root,
        train_images=SMOKE_TRAIN_IMAGES,
        val_images=SMOKE_VAL_IMAGES,
        seed=SMOKE_SEED,
        clean_output=True,
    )
    print('subset:', subset_report)

    val_report = validate_visdrone_yolo_structure(subset_root, splits=('train', 'val'))
    print('subset valid:', val_report['is_valid'])

    smoke_stats_dir = PROJECT_ROOT / 'artifacts' / 'smoke' / 'stats'
    analyzer_cfg = DatasetAnalyzerConfig(
        small_max_area=float(project_cfg['analysis']['small_max_area']),
        medium_max_area=float(project_cfg['analysis']['medium_max_area']),
        tiny_max_area=float(project_cfg['analysis']['tiny_max_area']),
        image_extensions=tuple(project_cfg['dataset'].get('image_extensions', ['.jpg', '.jpeg', '.png', '.bmp'])),
        generate_plots=False,
    )
    smoke_stats = analyze_dataset(
        dataset_root=subset_root,
        output_dir=smoke_stats_dir,
        splits=SMOKE_SPLITS,
        config=analyzer_cfg,
    )

    smoke_policy_dir = PROJECT_ROOT / 'artifacts' / 'smoke' / 'policy'
    rule_cfg = RuleEngineConfig.from_project_config(project_cfg)
    smoke_policy, smoke_decision_report = generate_policy_from_stats(smoke_stats, cfg=rule_cfg)
    smoke_paths = save_policy_artifacts(smoke_policy, smoke_decision_report, output_dir=smoke_policy_dir)

    required_files = [
        smoke_stats_dir / 'dataset_stats.json',
        smoke_stats_dir / 'dataset_stats.csv',
        smoke_policy_dir / 'policy_adaptive.json',
        smoke_policy_dir / 'policy_adaptive.yaml',
        smoke_policy_dir / 'decision_report.json',
    ]
    for p in required_files:
        assert p.exists(), f'Missing artifact: {p}'

    print('\n[OK] Smoke check passed')
    print('dataset_stats:', smoke_stats_dir / 'dataset_stats.json')
    print('policy_json:', smoke_policy_dir / 'policy_adaptive.json')
    print('decision_report:', smoke_policy_dir / 'decision_report.json')
    print('selected albu transforms:', [x['name'] for x in smoke_policy['albumentations_spec']])
    print('ultralytics args:', json.dumps(smoke_policy['ultralytics_args'], ensure_ascii=False, indent=2))
else:
    print('Smoke is skipped. Set RUN_SMOKE=True to enable.')


In [ ]:
# Step 1-2 (optional full run): validate and analyze dataset.
# Default is quality-oriented: use full dataset for stronger policy/training signal.
RUN_FULL_ANALYSIS = True
USE_SUBSET_FOR_FULL_PIPELINE = False
SUBSET_TRAIN_IMAGES = 5000
SUBSET_VAL_IMAGES = 1000
SUBSET_SEED = 42
GENERATE_PLOTS_FOR_FULL_ANALYSIS = False

if RUN_FULL_ANALYSIS:
    from src.data.subset_builder import build_yolo_subset

    pipeline_dataset_root = dataset_root

    if USE_SUBSET_FOR_FULL_PIPELINE:
        subset_root = PROJECT_ROOT / 'datasets' / 'visdrone_full_subset'
        subset_report = build_yolo_subset(
            dataset_root=dataset_root,
            output_root=subset_root,
            train_images=SUBSET_TRAIN_IMAGES,
            val_images=SUBSET_VAL_IMAGES,
            seed=SUBSET_SEED,
            clean_output=True,
        )
        pipeline_dataset_root = subset_root
        print('[INFO] Full pipeline subset prepared:', subset_report)
    else:
        print('[INFO] Using full dataset for full pipeline run.')

    print('[INFO] Analysis dataset root:', pipeline_dataset_root)

    validation_report = validate_visdrone_yolo_structure(
        dataset_root=pipeline_dataset_root,
        splits=splits,
        image_extensions=image_extensions,
    )
    save_validation_report(validation_report, stats_dir / 'validation_report.json')
    print('Validation is_valid =', validation_report['is_valid'])

    analyzer_cfg = DatasetAnalyzerConfig(
        small_max_area=float(project_cfg['analysis']['small_max_area']),
        medium_max_area=float(project_cfg['analysis']['medium_max_area']),
        tiny_max_area=float(project_cfg['analysis']['tiny_max_area']),
        image_extensions=image_extensions,
        generate_plots=GENERATE_PLOTS_FOR_FULL_ANALYSIS,
        show_progress=True,
        progress_log_every=200,
    )
    stats = analyze_dataset(
        dataset_root=pipeline_dataset_root,
        output_dir=stats_dir,
        splits=splits,
        config=analyzer_cfg,
    )
    print('Stats saved to:', stats_dir)
else:
    print('Full analysis is skipped. Set RUN_FULL_ANALYSIS=True to enable.')



In [ ]:
# Step 3 (optional full run): generate adaptive policy from full stats.
RUN_FULL_POLICY = True

if RUN_FULL_POLICY:
    rule_cfg = RuleEngineConfig.from_project_config(project_cfg)
    policy, decision_report = generate_policy_from_stats(stats, cfg=rule_cfg)
    paths = save_policy_artifacts(policy=policy, decision_report=decision_report, output_dir=policy_dir)

    print('Policy JSON:', paths['policy_json'])
    print('Policy YAML:', paths['policy_yaml'])
    print('Decision report:', paths['decision_report'])
    print('Fired rules:', len(decision_report.get('fired_rules', [])))
else:
    print('Full policy generation is skipped. Set RUN_FULL_POLICY=True to enable.')


In [ ]:
# Step 3.5: optional dataset integrity check.
# By default we do NOT rebuild from raw if a ready YOLO dataset is already available.
from pathlib import Path

from src.data.visdrone_manager import prepare_visdrone_auto, validate_visdrone_yolo_structure

RAW_ROOT = Path('/content/drive/MyDrive/datasets/visdrone_raw')
YOLO_ROOT = Path('/content/drive/MyDrive/datasets/visdrone_yolo')
FORCE_REBUILD_FROM_RAW = False
AUTO_REBUILD_IF_SMALL_TRAIN = False
MIN_EXPECTED_FULL_TRAIN_IMAGES = 3000
SYNC_PIPELINE_DATASET_ROOT = True

# Prefer dataset_root resolved in bootstrap cell.
active_root = Path(globals().get('dataset_root', YOLO_ROOT))

def _summarize_yolo(root: Path):
    report = validate_visdrone_yolo_structure(root, splits=('train', 'val'))
    train_n = report['splits']['train']['num_images']
    val_n = report['splits']['val']['num_images']
    print('YOLO root:', root)
    print('is_valid:', report['is_valid'])
    print('train images:', train_n)
    print('val images:', val_n)
    return report, train_n, val_n

if FORCE_REBUILD_FROM_RAW:
    print(f'[INFO] Rebuilding YOLO dataset from raw: {RAW_ROOT} -> {YOLO_ROOT}')
    prepare_visdrone_auto(raw_visdrone_root=RAW_ROOT, output_root=YOLO_ROOT)
    active_root = YOLO_ROOT

report, train_n, val_n = _summarize_yolo(active_root)

if AUTO_REBUILD_IF_SMALL_TRAIN and train_n < MIN_EXPECTED_FULL_TRAIN_IMAGES and RAW_ROOT.exists():
    print(
        f"[WARN] train split looks too small ({train_n}). "
        f"Trying auto-rebuild from raw root: {RAW_ROOT}"
    )
    prepare_visdrone_auto(raw_visdrone_root=RAW_ROOT, output_root=YOLO_ROOT)
    active_root = YOLO_ROOT
    report, train_n, val_n = _summarize_yolo(active_root)

assert train_n > 0 and val_n > 0, 'Train/Val images are empty.'
if train_n < MIN_EXPECTED_FULL_TRAIN_IMAGES:
    print(
        f"[WARN] train split is still small ({train_n}). "
        'Full VisDrone train is expected to be larger.'
    )

# Keep notebook-wide roots consistent for downstream cells.
dataset_root = active_root
if SYNC_PIPELINE_DATASET_ROOT:
    pipeline_dataset_root = dataset_root
print('[INFO] dataset_root set to:', dataset_root)



In [ ]:
# Step 4 (optional): run training suite with speed/quality presets and Drive sync.
from pathlib import Path
from dataclasses import replace
import shutil
import yaml

RUN_TRAINING = True

# Runtime profile:
# - fast: quick sanity check (~10-20 min on T4)
# - balanced: practical dev run (~30-60 min)
# - hour_safe: around ~60-90 min, usually best trade-off
# - hour_full: slower, full validation each epoch
# - max_quality: long run for stronger final metrics
TRAIN_PROFILE = 'hour_safe'

# Which modes to train. Use ['adaptive'] for the fastest practical experiment.
TRAIN_MODES = ['baseline', 'manual', 'adaptive']  # fair comparison under identical conditions
RUN_ABLATION = False
MODEL_OVERRIDE = 'yolo11s.pt'  # основной весовой файл для экспериментов

SAVE_RUNS_TO_DRIVE = True
RUNS_LOCAL_DIR = PROJECT_ROOT / 'runs' / 'visdrone'
RUNS_DRIVE_DIR = globals().get('DRIVE_ARTIFACTS_ROOT', Path('/content/drive/MyDrive/experiments/small_objects_auto_aug/visdrone')) / 'runs'

if RUN_TRAINING:
    import importlib
    import src.training.train_runner as train_runner_module

    from src.utils.io import load_yaml, load_json
    from src.augmentation.policy_to_ultralytics import AugmentationPolicy

    importlib.reload(train_runner_module)
    TrainRunConfig = train_runner_module.TrainRunConfig
    run_train_mode = train_runner_module.run_train_mode
    run_mvp_training_suite = train_runner_module.run_mvp_training_suite

    train_dataset_root = globals().get('pipeline_dataset_root', dataset_root)

    def _count_split_images(root: Path, split: str) -> int:
        exts = {'.jpg', '.jpeg', '.png', '.bmp'}
        images_dir = root / 'images' / split
        if not images_dir.exists():
            return 0
        return sum(1 for p in images_dir.iterdir() if p.is_file() and p.suffix.lower() in exts)

    train_n = _count_split_images(train_dataset_root, 'train')
    val_n = _count_split_images(train_dataset_root, 'val')
    print(f'[INFO] Training dataset root: {train_dataset_root}')
    print(f'[INFO] train images: {train_n}, val images: {val_n}')
    assert train_n > 0 and val_n > 0, 'Training dataset has empty train/val split.'

    profile_map = {
        'fast': {
            'epochs': 25,
            'imgsz': 960,
            'batch': 6,
            'workers': 2,
            'fraction': 1.0,
            'multi_scale': False,
            'val': True,
            'cache': 'disk',
            'patience': 20,
        },
        'balanced': {
            'epochs': 25,
            'imgsz': 960,
            'batch': 6,
            'workers': 2,
            'fraction': 1.0,
            'multi_scale': False,
            'val': True,
            'cache': 'disk',
            'patience': 30,
        },
        'hour_safe': {
            'epochs': 25,
            'imgsz': 960,
            'batch': 6,
            'workers': 2,
            'fraction': 1.0,
            'multi_scale': False,
            'val': True,
            'cache': 'disk',
            'patience': 40,
        },
        'hour_full': {
            'epochs': 25,
            'imgsz': 960,
            'batch': 6,
            'workers': 2,
            'fraction': 1.0,
            'multi_scale': False,
            'val': True,
            'cache': 'disk',
            'patience': 60,
        },
        'max_quality': {
            'epochs': 25,
            'imgsz': 960,
            'batch': 6,
            'workers': 2,
            'fraction': 1.0,
            'multi_scale': False,
            'val': True,
            'cache': 'disk',
            'patience': 90,
        },
    }
    assert TRAIN_PROFILE in profile_map, f'Unknown TRAIN_PROFILE: {TRAIN_PROFILE}'
    prof = profile_map[TRAIN_PROFILE]
    print(f'[INFO] Training profile: {TRAIN_PROFILE} -> {prof}')

    runtime_data_yaml = PROJECT_ROOT / 'artifacts' / 'runtime_visdrone.yaml'
    runtime_data_yaml.parent.mkdir(parents=True, exist_ok=True)

    names = project_cfg['dataset']['visdrone_classes']
    data_cfg = {
        'path': str(train_dataset_root),
        'train': 'images/train',
        'val': True,
        'test': 'images/test',
        'names': {i: n for i, n in enumerate(names)},
    }
    runtime_data_yaml.write_text(
        yaml.safe_dump(data_cfg, sort_keys=False, allow_unicode=True),
        encoding='utf-8',
    )
    print(f'[INFO] Runtime data.yaml: {runtime_data_yaml}')

    runtime_baseline_yaml = PROJECT_ROOT / 'artifacts' / 'runtime_baseline.yaml'
    runtime_manual_yaml = PROJECT_ROOT / 'artifacts' / 'runtime_manual.yaml'

    baseline_args = {
        'mosaic': 0.9,
        'close_mosaic': 15,
        'mixup': 0.0,
        'cutmix': 0.0,
        'hsv_h': 0.015,
        'hsv_s': 0.7,
        'hsv_v': 0.45,
        'degrees': 2.0,
        'translate': 0.08,
        'scale': 0.55,
        'perspective': 0.0,
        'fliplr': 0.5,
        'flipud': 0.0,
    }
    manual_args = {
        'mosaic': 0.85,
        'close_mosaic': 15,
        'mixup': 0.0,
        'cutmix': 0.0,
        'hsv_h': 0.015,
        'hsv_s': 0.6,
        'hsv_v': 0.5,
        'degrees': 5.0,
        'translate': 0.08,
        'scale': 0.45,
        'perspective': 0.0,
        'fliplr': 0.5,
        'flipud': 0.0,
    }
    runtime_baseline_yaml.write_text(yaml.safe_dump(baseline_args, sort_keys=False), encoding='utf-8')
    runtime_manual_yaml.write_text(yaml.safe_dump(manual_args, sort_keys=False), encoding='utf-8')

    full_policy = policy_dir / 'policy_adaptive.json'
    smoke_policy = PROJECT_ROOT / 'artifacts' / 'smoke' / 'policy' / 'policy_adaptive.json'
    adaptive_policy_json = full_policy if full_policy.exists() else smoke_policy
    assert adaptive_policy_json.exists(), f'Policy JSON not found: {adaptive_policy_json}'

    model_name = str(MODEL_OVERRIDE or project_cfg['training']['model'])

    cfg_kwargs = dict(
        data_yaml=str(runtime_data_yaml),
        model=model_name,
        epochs=int(prof['epochs']),
        imgsz=int(prof['imgsz']),
        batch=int(prof['batch']),
        device=0,
        workers=int(prof['workers']),
        fraction=float(prof['fraction']),
        project_dir=str(RUNS_LOCAL_DIR),
        seed=int(project_cfg['project']['seed']),
        deterministic=bool(project_cfg['project']['deterministic']),
        rect=bool(project_cfg['training'].get('rect', False)),
        multi_scale=bool(prof['multi_scale']),
        baseline_disable_albumentations=bool(project_cfg['training'].get('baseline_disable_albumentations', True)),
        val=bool(prof['val']),
        cache=prof['cache'],
        save_period=-1,
        patience=int(prof['patience']),
    )
    if hasattr(TrainRunConfig, '__dataclass_fields__') and 'plots' in TrainRunConfig.__dataclass_fields__:
        cfg_kwargs['plots'] = False

    train_cfg = TrainRunConfig(**cfg_kwargs)

    mode_overrides = {
        mode: {
            'val': True,
            'cache': prof['cache'],
            'patience': int(prof['patience']),
        }
        for mode in TRAIN_MODES
    }

    requested_default_modes = TRAIN_MODES == ['baseline', 'manual', 'adaptive']
    if requested_default_modes:
        run_dirs = run_mvp_training_suite(
            config=train_cfg,
            baseline_yaml_path=runtime_baseline_yaml,
            manual_yaml_path=runtime_manual_yaml,
            adaptive_policy_json_path=adaptive_policy_json,
            run_ablation=RUN_ABLATION,
            mode_overrides=mode_overrides,
        )
    else:
        base_args = load_yaml(runtime_baseline_yaml)
        man_args = load_yaml(runtime_manual_yaml)
        adaptive_payload = load_json(adaptive_policy_json)
        adaptive_policy = AugmentationPolicy(payload=adaptive_payload)

        mode_to_args = {
            'baseline': base_args,
            'manual': man_args,
            'adaptive': adaptive_policy.get_ultralytics_train_args(),
        }
        mode_to_custom_aug = {
            'baseline': None,
            'manual': None,
            'adaptive': adaptive_policy.get_albumentations_transforms(seed=train_cfg.seed),
        }

        run_dirs = {}
        for mode in TRAIN_MODES:
            if mode not in mode_to_args:
                raise ValueError(f'Unsupported mode in TRAIN_MODES: {mode}')
            mode_cfg = replace(train_cfg, **mode_overrides.get(mode, {}))
            run_dir = run_train_mode(
                mode=mode,
                config=mode_cfg,
                mode_args=mode_to_args[mode],
                custom_augmentations=mode_to_custom_aug[mode],
            )
            run_dirs[mode] = run_dir.as_posix()

    print('[INFO] Local run dirs:')
    print(run_dirs)

    if SAVE_RUNS_TO_DRIVE:
        RUNS_DRIVE_DIR.mkdir(parents=True, exist_ok=True)
        synced = {}
        for mode, local_dir_str in run_dirs.items():
            local_dir = Path(local_dir_str)
            if not local_dir.exists():
                continue
            target_dir = RUNS_DRIVE_DIR / mode
            shutil.copytree(local_dir, target_dir, dirs_exist_ok=True)
            synced[mode] = target_dir.as_posix()
        print('[INFO] Synced run dirs to Drive:')
        print(synced)

    latest_run_dirs = run_dirs
else:
    print('Training is skipped. Set RUN_TRAINING=True to enable.')


In [ ]:
# Step 5 (optional): inference on val + COCO conversion + COCOeval + Drive sync.
RUN_EVAL = True
USE_TTA = True
EVAL_IMGSZ = 960
SAVE_EVAL_TO_DRIVE = True

if RUN_EVAL:
    import pandas as pd
    import shutil
    from pathlib import Path
    from ultralytics import YOLO

    from src.evaluation.coco_converter import convert_yolo_gt_to_coco, convert_yolo_pred_txt_to_coco
    from src.evaluation.coco_eval_runner import run_coco_eval
    from src.evaluation.metrics_report import build_markdown_report

    eval_dataset_root = globals().get('pipeline_dataset_root', dataset_root)
    print(f'[INFO] Eval dataset root: {eval_dataset_root}')

    eval_dir = PROJECT_ROOT / 'artifacts' / 'eval'
    eval_dir.mkdir(parents=True, exist_ok=True)

    def _pick_best_run_from_dirs(run_dirs_map: dict[str, str]):
        best = None
        for name, run_dir_str in run_dirs_map.items():
            run_dir = Path(run_dir_str)
            weights = run_dir / 'weights' / 'best.pt'
            results_csv = run_dir / 'results.csv'
            if not weights.exists():
                continue

            score = -1.0
            if results_csv.exists():
                try:
                    df = pd.read_csv(results_csv)
                    if 'metrics/mAP50-95(B)' in df.columns:
                        score = float(df['metrics/mAP50-95(B)'].max())
                except Exception:
                    score = -1.0

            if best is None or score > best['score']:
                best = {'name': name, 'weights': weights, 'score': score, 'run_dir': run_dir}
        return best

    run_dirs_map = globals().get('latest_run_dirs', None)
    if not run_dirs_map:
        fallback = {}
        for name in ['adaptive', 'manual', 'baseline', 'adaptive_no_mosaic', 'adaptive_no_custom_albu']:
            d = (PROJECT_ROOT / 'runs' / 'visdrone' / name)
            if d.exists():
                fallback[name] = d.as_posix()
        run_dirs_map = fallback

    best = _pick_best_run_from_dirs(run_dirs_map)
    assert best is not None, 'No trained weights found. Run training first.'

    best_weights = best['weights']
    run_used = best['name']
    print(f"[INFO] Using run '{run_used}' for eval (best val mAP50-95={best['score']:.5f})")

    pred_project = PROJECT_ROOT / 'runs' / 'eval_predict'
    model = YOLO(str(best_weights))
    model.predict(
        source=str(eval_dataset_root / 'images' / 'val'),
        device=0,
        imgsz=EVAL_IMGSZ,
        conf=0.001,
        iou=0.6,
        max_det=500,
        augment=USE_TTA,
        save=False,
        save_txt=True,
        save_conf=True,
        project=str(pred_project),
        name=run_used,
        exist_ok=True,
        verbose=False,
    )

    pred_labels_dir = pred_project / run_used / 'labels'
    assert pred_labels_dir.exists(), f'Prediction labels not found: {pred_labels_dir}'

    coco_gt_path = eval_dir / 'coco_gt.json'
    coco_dt_path = eval_dir / 'coco_dt.json'

    convert_yolo_gt_to_coco(
        images_dir=eval_dataset_root / 'images' / 'val',
        labels_dir=eval_dataset_root / 'labels' / 'val',
        class_names=project_cfg['dataset']['visdrone_classes'],
        output_path=coco_gt_path,
        image_extensions=image_extensions,
    )
    convert_yolo_pred_txt_to_coco(
        pred_labels_dir=pred_labels_dir,
        images_dir=eval_dataset_root / 'images' / 'val',
        output_path=coco_dt_path,
        image_extensions=image_extensions,
    )

    metrics = run_coco_eval(
        coco_gt_path=coco_gt_path,
        coco_dt_path=coco_dt_path,
        output_path=eval_dir / 'coco_eval.json',
        use_tiny_eval=True,
        tiny_max_area=float(project_cfg['analysis']['tiny_max_area']),
    )

    report_path = build_markdown_report(
        {f'{run_used}_eval': metrics},
        PROJECT_ROOT / 'artifacts' / 'reports' / 'mvp_report.md'
    )

    print('Run used:', run_used)
    print(metrics)
    print('Report:', report_path)

    if SAVE_EVAL_TO_DRIVE:
        drive_artifacts_root = globals().get('DRIVE_ARTIFACTS_ROOT', Path('/content/drive/MyDrive/experiments/small_objects_auto_aug/visdrone'))
        drive_eval_dir = drive_artifacts_root / 'eval'
        drive_reports_dir = drive_artifacts_root / 'reports'
        drive_eval_dir.mkdir(parents=True, exist_ok=True)
        drive_reports_dir.mkdir(parents=True, exist_ok=True)

        shutil.copy2(eval_dir / 'coco_gt.json', drive_eval_dir / 'coco_gt.json')
        shutil.copy2(eval_dir / 'coco_dt.json', drive_eval_dir / 'coco_dt.json')
        shutil.copy2(eval_dir / 'coco_eval.json', drive_eval_dir / 'coco_eval.json')
        if Path(report_path).exists():
            shutil.copy2(report_path, drive_reports_dir / Path(report_path).name)

        print('[INFO] Eval artifacts synced to Drive:')
        print(' -', drive_eval_dir)
        print(' -', drive_reports_dir)
else:
    print('Evaluation is skipped. Set RUN_EVAL=True to enable.')


In [ ]:
# Step X: unified monitoring export (train + val metrics history).
from pathlib import Path
import shutil
import pandas as pd

# This cell builds reusable CSV artifacts for later plots in thesis:
# 1) full epoch history across runs
# 2) last-epoch snapshot per run
# It includes both train-side metrics (losses) and val-side metrics (precision/recall/mAP).

PROJECT_ROOT_RESOLVED = Path(globals().get('PROJECT_ROOT', Path.cwd()))


def _resolve_runtime_cfg_path() -> Path | None:
    candidate_names = ['runtime_cfg_path']
    for name in candidate_names:
        value = globals().get(name)
        if value is not None:
            p = Path(value)
            if p.exists():
                return p

    hints = [
        PROJECT_ROOT_RESOLVED / 'artifacts' / 'dota_runtime_config.yaml',
        PROJECT_ROOT_RESOLVED / 'artifacts' / 'xview_runtime_config.yaml',
        PROJECT_ROOT_RESOLVED / 'artifacts' / 'coco_small_runtime_config.yaml',
        PROJECT_ROOT_RESOLVED / 'artifacts' / 'dota_scaleaware_vs_adaptive_runtime.yaml',
        PROJECT_ROOT_RESOLVED / 'artifacts' / 'runtime_config.yaml',
    ]
    for p in hints:
        if p.exists():
            return p
    return None


def _load_cfg(cfg_path: Path | None):
    if cfg_path is None:
        return None
    try:
        import yaml
        return yaml.safe_load(cfg_path.read_text(encoding='utf-8'))
    except Exception:
        return None


def _resolve_runs_root(cfg: dict | None) -> Path:
    name_candidates = [
        'RUNS_ROOT',
        'RUNS_DRIVE_DIR',
        'DOTA_EXPERIMENT_RUNS',
        'COCO_EXPERIMENT_RUNS',
        'XVIEW_EXPERIMENT_RUNS',
    ]
    for name in name_candidates:
        value = globals().get(name)
        if value is not None:
            p = Path(value)
            if p.exists():
                return p

    if isinstance(cfg, dict):
        p = cfg.get('training', {}).get('project_dir')
        if p:
            rp = Path(p)
            if rp.exists():
                return rp

    fallback = PROJECT_ROOT_RESOLVED / 'runs'
    fallback.mkdir(parents=True, exist_ok=True)
    return fallback


def _resolve_drive_artifacts_root() -> Path | None:
    candidates = [
        globals().get('ARTIFACTS_ROOT'),
        globals().get('DOTA_EXPERIMENT_ARTIFACTS'),
        globals().get('COCO_EXPERIMENT_ARTIFACTS'),
        globals().get('XVIEW_EXPERIMENT_ARTIFACTS'),
        globals().get('ARTIFACTS_DRIVE_DIR'),
        globals().get('DRIVE_ARTIFACTS_ROOT'),
    ]
    for item in candidates:
        if item is None:
            continue
        p = Path(item)
        try:
            p.mkdir(parents=True, exist_ok=True)
            return p
        except Exception:
            continue
    return None


cfg_path = _resolve_runtime_cfg_path()
cfg = _load_cfg(cfg_path)
runs_root = _resolve_runs_root(cfg)

monitor_dir = PROJECT_ROOT_RESOLVED / 'artifacts' / 'monitoring'
monitor_dir.mkdir(parents=True, exist_ok=True)

results_files = sorted(runs_root.rglob('results.csv'))
if not results_files:
    raise FileNotFoundError(f'No results.csv found under runs root: {runs_root}')

frames = []
for csv_path in results_files:
    run_dir = csv_path.parent
    run_name = run_dir.name
    try:
        df = pd.read_csv(csv_path)
    except Exception as exc:
        print(f'[WARN] failed to read {csv_path}: {exc}')
        continue
    if df.empty:
        continue

    if 'epoch' not in df.columns:
        df['epoch'] = list(range(len(df)))

    df['run_name'] = run_name
    df['run_dir'] = run_dir.as_posix()
    frames.append(df)

if not frames:
    raise RuntimeError('No readable non-empty results.csv files were found.')

history_wide = pd.concat(frames, ignore_index=True)

# Keep numeric metric columns + id columns.
id_cols = ['run_name', 'run_dir', 'epoch']
metric_cols = [
    c for c in history_wide.columns
    if c not in id_cols and pd.api.types.is_numeric_dtype(history_wide[c])
]
history_wide = history_wide[id_cols + metric_cols]

history_wide_path = monitor_dir / 'metrics_history_wide.csv'
history_wide.to_csv(history_wide_path, index=False)

# Last-epoch snapshot per run (useful for quick comparison tables).
last_rows = history_wide.sort_values(['run_name', 'epoch']).groupby('run_name', as_index=False).tail(1)
last_rows = last_rows.sort_values('run_name').reset_index(drop=True)
last_rows_path = monitor_dir / 'metrics_last_epoch.csv'
last_rows.to_csv(last_rows_path, index=False)

# Build long-format table for plotting (run/epoch/metric/value).
history_long = history_wide.melt(
    id_vars=['run_name', 'run_dir', 'epoch'],
    value_vars=metric_cols,
    var_name='metric',
    value_name='value',
)

def _metric_split(metric_name: str) -> str:
    m = metric_name.lower()
    if m.startswith('metrics/') or m.startswith('val/'):
        return 'val'
    if m.startswith('train/') or m.endswith('_loss'):
        return 'train'
    if 'loss' in m:
        return 'train'
    return 'other'

history_long['split'] = history_long['metric'].map(_metric_split)
history_long_path = monitor_dir / 'metrics_history_long.csv'
history_long.to_csv(history_long_path, index=False)

# Quick console-friendly view for thesis workflow.
interesting = [
    'train/box_loss', 'train/cls_loss', 'train/dfl_loss',
    'box_loss', 'cls_loss', 'dfl_loss',
    'metrics/precision(B)', 'metrics/recall(B)',
    'metrics/mAP50(B)', 'metrics/mAP50-95(B)',
]
show_cols = ['run_name', 'epoch'] + [c for c in interesting if c in last_rows.columns]
print('[OK] history_wide:', history_wide_path)
print('[OK] history_long:', history_long_path)
print('[OK] last_epoch:', last_rows_path)
print('\nLast-epoch snapshot (train + val):')
print(last_rows[show_cols].to_string(index=False))

# Optional Drive sync for artifacts.
drive_artifacts_root = _resolve_drive_artifacts_root()
if drive_artifacts_root is not None:
    drive_monitor_dir = drive_artifacts_root / 'monitoring'
    drive_monitor_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(history_wide_path, drive_monitor_dir / history_wide_path.name)
    shutil.copy2(history_long_path, drive_monitor_dir / history_long_path.name)
    shutil.copy2(last_rows_path, drive_monitor_dir / last_rows_path.name)
    print('[OK] Monitoring CSVs synced to Drive:', drive_monitor_dir)


In [ ]:
# Step Y: plot preview for train/val metrics (for thesis figures draft).
from pathlib import Path
import shutil
import pandas as pd
import matplotlib.pyplot as plt

RUN_PLOT_PREVIEW = True

if RUN_PLOT_PREVIEW:
    project_root = Path(globals().get('PROJECT_ROOT', Path.cwd()))
    monitor_dir = project_root / 'artifacts' / 'monitoring'
    history_long_path = monitor_dir / 'metrics_history_long.csv'

    if not history_long_path.exists():
        raise FileNotFoundError(
            f'Missing monitoring history: {history_long_path}. '
            'Run the monitoring export cell first.'
        )

    df = pd.read_csv(history_long_path)
    if df.empty:
        raise RuntimeError('metrics_history_long.csv is empty.')

    # Keep numeric rows only.
    df = df[pd.to_numeric(df['value'], errors='coerce').notna()].copy()
    df['value'] = df['value'].astype(float)

    plots_dir = monitor_dir / 'plots'
    plots_dir.mkdir(parents=True, exist_ok=True)

    # Metrics that are usually useful for diploma report.
    metric_candidates = [
        'metrics/mAP50-95(B)',
        'metrics/mAP50(B)',
        'metrics/precision(B)',
        'metrics/recall(B)',
        'train/box_loss',
        'train/cls_loss',
        'train/dfl_loss',
        'box_loss',
        'cls_loss',
        'dfl_loss',
    ]
    metrics = [m for m in metric_candidates if m in set(df['metric'].unique())]

    if not metrics:
        raise RuntimeError('No known metrics found for plotting in metrics_history_long.csv.')

    created = []

    # 1) Per-metric comparison across runs.
    for metric in metrics:
        sub = df[df['metric'] == metric].copy()
        if sub.empty:
            continue

        plt.figure(figsize=(9, 5))
        for run_name, grp in sub.groupby('run_name'):
            grp = grp.sort_values('epoch')
            plt.plot(grp['epoch'], grp['value'], marker='o', markersize=2, linewidth=1.6, label=str(run_name))

        plt.title(f'{metric} vs epoch')
        plt.xlabel('epoch')
        plt.ylabel(metric)
        plt.grid(alpha=0.3)
        plt.legend(loc='best', fontsize=8)
        out = plots_dir / f"metric_{metric.replace('/', '_').replace('(', '').replace(')', '')}.png"
        plt.tight_layout()
        plt.savefig(out, dpi=160)
        plt.close()
        created.append(out)

    # 2) Overview: best val metrics by run (last epoch).
    last_rows = (
        df.sort_values(['run_name', 'epoch'])
          .groupby(['run_name', 'metric'], as_index=False)
          .tail(1)
    )

    val_metrics_for_bar = [m for m in ['metrics/mAP50-95(B)', 'metrics/mAP50(B)', 'metrics/precision(B)', 'metrics/recall(B)'] if m in set(last_rows['metric'])]
    if val_metrics_for_bar:
        pivot = (
            last_rows[last_rows['metric'].isin(val_metrics_for_bar)]
            .pivot_table(index='run_name', columns='metric', values='value', aggfunc='mean')
            .sort_index()
        )

        ax = pivot.plot(kind='bar', figsize=(10, 5), rot=25)
        ax.set_title('Final val metrics by run (last epoch)')
        ax.set_ylabel('value')
        ax.grid(axis='y', alpha=0.3)
        out_bar = plots_dir / 'final_val_metrics_bar.png'
        plt.tight_layout()
        plt.savefig(out_bar, dpi=160)
        plt.close()
        created.append(out_bar)

    print('[OK] Plot files created:')
    for p in created:
        print(' -', p)

    # Optional Drive sync.
    drive_candidates = [
        globals().get('ARTIFACTS_ROOT'),
        globals().get('DOTA_EXPERIMENT_ARTIFACTS'),
        globals().get('COCO_EXPERIMENT_ARTIFACTS'),
        globals().get('XVIEW_EXPERIMENT_ARTIFACTS'),
        globals().get('ARTIFACTS_DRIVE_DIR'),
        globals().get('DRIVE_ARTIFACTS_ROOT'),
    ]
    drive_root = None
    for c in drive_candidates:
        if c is None:
            continue
        try:
            p = Path(c)
            p.mkdir(parents=True, exist_ok=True)
            drive_root = p
            break
        except Exception:
            continue

    if drive_root is not None:
        drive_plots_dir = drive_root / 'monitoring' / 'plots'
        drive_plots_dir.mkdir(parents=True, exist_ok=True)
        for p in created:
            shutil.copy2(p, drive_plots_dir / p.name)
        print('[OK] Plot files synced to Drive:', drive_plots_dir)
else:
    print('Plot preview skipped. Set RUN_PLOT_PREVIEW=True to enable.')
